# 06 - Preparação para classificação de falhas

**Objetivo:** Criar janelas e características para normal, travado e ruído excessivo.

> Execute as células na ordem. As saídas são gravadas nas pastas do projeto.

In [ ]:
from pathlib import Path
import sys, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Localiza a raiz do projeto independentemente de o notebook ser aberto pela pasta raiz ou notebooks.
CANDIDATOS=[Path.cwd(), Path.cwd().parent]
ROOT=next((p.resolve() for p in CANDIDATOS if (p/'dados').exists() and (p/'notebooks').exists()),None)
if ROOT is None: raise FileNotFoundError('Abra o VS Code na pasta projeto_soft_sensor.')
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from src.funcoes import *
sns.set_theme(style='whitegrid')
print('Raiz:',ROOT)


## Carregamento e classes

In [ ]:
d=carregar_csv(ROOT/'dados/02_intermediarios/dados_com_flags_qualidade.csv')
mapa={'sensor_travado_temperatura':1,'ruido_excessivo_temperatura':2}; d['classe_alvo']=d.evento_referencia.map(mapa).fillna(0).astype(int)
# Regras físicas e comunicação são tratadas fora do classificador; excluímos tais pontos das janelas de treino.
d=d[~d.evento_referencia.isin(['valor_fisicamente_impossivel','falha_de_comunicacao','dado_faltante_aleatorio','pico_isolado_desconhecido'])].copy()
print(d.classe_alvo.value_counts())

## Extração de janelas

In [ ]:
bases=[]
for janela in [6,12,24]:
    f=extrair_janelas(d,janela=janela,passo=3); f['janela_amostras']=janela; bases.append(f)
features=pd.concat(bases,ignore_index=True); display(features.groupby(['janela_amostras','classe']).size().unstack(fill_value=0))

## Partição cronológica e exportação

In [ ]:
# Partição por tempo; eventos permanecem inteiros porque os eventos foram posicionados em blocos distintos.
q1=features.timestamp_fim.quantile(.60); q2=features.timestamp_fim.quantile(.80); features['particao']=np.where(features.timestamp_fim<=q1,'treino',np.where(features.timestamp_fim<=q2,'validacao','teste'))
out=ROOT/'dados/03_processados/base_classificacao_janelas.csv'; features.to_csv(out,index=False,encoding='utf-8-sig'); print('Salvo:',out,features.shape); display(pd.crosstab([features.particao,features.janela_amostras],features.classe))

## Conclusões da etapa

Execute todas as células, registre as observações exibidas e confirme os arquivos gerados antes de avançar para o notebook seguinte.